# More Than A Hack 2027 – Kontaktbasierte Klassifikation testen

**Frage:** Verbessert eine Teamzuordnung anhand der Kontaktkandidaten aus Notebook 03 die öffentliche H/A/D-Klassifikation gegenüber `baseline_smooth50`?

**Ein isolierter erster Versuch:** Die Unterbrechungserkennung D bleibt exakt diejenige der offiziellen Baseline. Wir verändern nur die Zuordnung zwischen H und A. Ein Kontaktkandidat übernimmt den Besitz für das Team des nächsten beobachteten Spielers; diese Zuordnung bleibt bis zum nächsten Kandidaten oder einer Baseline-Unterbrechung bestehen. Vor dem ersten Kandidaten eines Live-Abschnitts liefert die Baseline die Zuordnung. Es gibt keine Parametersuche und keine weitere Glättung der neuen Vorhersagen.

**Wichtige Grenze:** Dieses Verfahren ist eine Heuristik aus Kontaktindizien, **keine validierte Erkennung bestätigter Ballkontrolle und kein trainierter IBC-Klassifikator des Papers**. Ein unkontrollierter gegnerischer Kontakt kann einen falschen Teamwechsel auslösen. Ob die Heuristik hilft, wird einmal mit unveränderter offizieller Evaluation geprüft.

Das Notebook ist unabhängig ausführbar. Es berechnet Merkmale und Vorhersagen für das komplette taktische Video, bevor die öffentliche Ground Truth eingelesen wird. Die Evaluation umfasst weiterhin nur die veröffentlichten fünf Minuten. Videos, manuelle Szenennotizen und Smart Tags werden nicht als Klassifikationsmerkmale verwendet.

**Ausführung:** Python mit `numpy`, `pandas`, `matplotlib`, `IPython` → **Restart Kernel → Run All**. Keine Downloads oder Installationen. Die eigene Analyse steht vollständig im Notebook; der unveränderte offizielle Validator/Evaluator wird aus dem festgelegten Referenzarchiv nur temporär entpackt. Notebook 02 und dessen Dateien bleiben unverändert. Ergebnisse entstehen unter `output/contact_classification/`.

In [ ]:
# Standardbibliothek
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
from tempfile import TemporaryDirectory
from importlib.metadata import version
import hashlib
import json
import subprocess
import sys
import xml.etree.ElementTree as ET

# Daten und Darstellung
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import display, Markdown

## 1. Versuchsplan vor der Bewertung festlegen

Alle Diagnoseparameter entsprechen Notebook 03. Die Rauschparameter sind illustrative Modellannahmen, nicht gemessene Trackinggenauigkeiten. Die Nähegrenze von 2 m ist unkalibriert; die Beschleunigungsgrenze von 4 m/s² stammt als Hinweis aus dem Paper. Keiner dieser Werte wird anhand der folgenden Scores angepasst.

Die publizierten Halbzeitbereiche verhindern, dass ein Kontaktzustand aus Vorlauf oder Halbzeitpause in die nächste Halbzeit getragen wird. Sie sind keine aus der GT abgeleiteten Besitzgrenzen. Außerhalb dieser Bereiche bleibt die Baseline erhalten. Quelle: [offizielle Datenbeschreibung](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/DATA.md).

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.xml").is_file())
META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
ARCHIVE = PROJECT_DIR / "reference/mtah2027_official_3de6b60.zip"
ARCHIVE_SHA256 = "019f497753d7f561d3fbb6048beef20b1bc1f244961db40d7c1914fa4ab76342"
COMMIT = "3de6b60aeabd2475d95879a651d38b72bc4fba00"
OUTPUT_DIR = PROJECT_DIR / "output/contact_classification"
SUBMISSION_DIR, EVALUATION_DIR = OUTPUT_DIR / "submissions", OUTPUT_DIR / "evaluation"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_NAME, METHOD_NAME = "baseline_smooth50", "contact_state_shared_D"
N_FRAMES, FPS, PITCH = 199319, 25, np.array([105.3, 68.0])
HALVES = [("1. Halbzeit", 11185, 90651), ("2. Halbzeit", 110808, 190197)]
MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2 = 0.25, 20.0
NEAR_DISTANCE_M, PEAK_ACCELERATION_M_S2 = 2.0, 4.0
MIN_RUN_SECONDS, EDGE_GUARD_SECONDS = 1.0, 0.4
LABEL_ORDER, COLORS = ["H", "A", "D"], ["#1764ab", "#c83e41", "#a0a0a0"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == ARCHIVE_SHA256, "Das Referenzarchiv wurde verändert."
print(f"Python {sys.version.split()[0]} | Referenz-Commit: {COMMIT}")
display(pd.Series({name: version(name) for name in ["numpy", "pandas", "matplotlib", "IPython"]}, name="Version").to_frame())
print("Ein festgelegter Vergleich, kein Training und keine Parametersuche.")

## 2. Vergleichsbaseline und Tracking laden

Die Baseline-Datei stammt aus dem eingefrorenen offiziellen Referenzarchiv. Notebook 02 hat bereits ihre exakte Reproduktion aus unseren Rohdaten bestätigt. Wir verwenden diese Vorhersagen hier als gemeinsame D-Erkennung und als Startwert, **nicht als Ground Truth**.

Die Spielerpositionen bleiben unverändert; nur der Ball wird für die Bewegungsmerkmale geglättet. Der rohe Geschwindigkeitskanal wird nicht verwendet. Die Berechnung läuft für das gesamte Video und ist nicht auf die sechs zuvor angesehenen Szenen beschränkt.

In [ ]:
with ZipFile(ARCHIVE) as official:
    ARCHIVE_ROOT = official.namelist()[0]
    baseline_table = pd.read_csv(BytesIO(official.read(ARCHIVE_ROOT + f"evaluation/results/{BASELINE_NAME}.csv")))
assert np.array_equal(baseline_table["frame"], np.arange(N_FRAMES)), "Baseline und Videoindex sind nicht framegleich."
baseline_labels = baseline_table["label"].to_numpy()

root = ET.parse(META_PATH).getroot()
team_nodes = root.findall("./Metadata/Teams/Team")
team_map = {t.get("id"): ("H" if i == 0 else "A", t.findtext("Name")) for i, t in enumerate(team_nodes)}
channel_to_track = {c.get("id"): c.get("playerId") for c in root.findall("./Metadata/PlayerChannels/PlayerChannel")}
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")
track_ids = [channel_to_track[g.find("PlayerChannelRef").get("playerChannelId")]
             for g in spec.findall("./SplitRegister/SplitRegister")]
tracks = pd.DataFrame([{"track_id": p.get("id"), "side": team_map[p.get("teamId")][0],
                        "team": team_map[p.get("teamId")][1]} for p in root.findall("./Metadata/Players/Player")]).set_index("track_id").loc[track_ids]
positions = np.empty((N_FRAMES, len(track_ids) + 1, 2), dtype=float)
frame_ids = np.empty(N_FRAMES, dtype=int)
with RAW_PATH.open() as handle:
    for row, line in enumerate(handle):
        raw_frame, players_raw, ball_raw = line.strip().split(":")
        frame_ids[row] = int(raw_frame) - 1
        positions[row] = np.fromstring(players_raw.replace(";", ",") + "," + ball_raw, sep=",").reshape(-1, 3)[:, :2] * PITCH
assert row + 1 == N_FRAMES and np.array_equal(frame_ids, np.arange(N_FRAMES)), "Tracking-Zeitachse stimmt nicht mit der Baseline überein."
player_xy, ball_xy = positions[:, :-1], positions[:, -1]
ball_seen = np.isfinite(ball_xy).all(axis=1)
print(f"{N_FRAMES:,} Frames, {len(tracks)} Track-Slots; Ball in {100 * ball_seen.mean():.1f} % des Gesamtvideos beobachtet.")
display(tracks.groupby(["side", "team"]).size().rename("Track-Slots").to_frame())

## 3. Dieselbe Interaktionsdiagnose wie in Notebook 03

Der RTS-Glätter wird unverändert aus Notebook 03 übernommen. Er verwendet zeitlich spätere Beobachtungen innerhalb desselben lückenlosen Abschnitts; das Verfahren ist offline. Es gibt keine Glättung oder Ableitung über Balllücken. Kurze Abschnitte und ihre äußeren 0,4 Sekunden liefern keine Kontaktkandidaten.

Im Unterschied zu Notebook 03 werden die Abschnitte nun über das ganze Video verarbeitet. Insbesondere am Ende des damaligen Fünf-Minuten-Fensters kann dadurch weiterer Beobachtungskontext verfügbar sein. Der Algorithmus und seine Parameter bleiben gleich.

In [ ]:
def smooth_ball_rts(points, fps, measurement_sigma, process_sigma):
    dt = 1 / fps
    transition = np.eye(4)
    transition[0, 2] = transition[1, 3] = dt
    observation = np.eye(2, 4)
    acceleration_map = np.r_[np.eye(2) * dt**2 / 2, np.eye(2) * dt]
    process_noise = acceleration_map @ acceleration_map.T * process_sigma**2
    measurement_noise = np.eye(2) * measurement_sigma**2
    state = np.r_[points[0], (points[1] - points[0]) / dt]
    covariance = np.diag([measurement_sigma**2, measurement_sigma**2, 100.0, 100.0])
    predicted = np.empty((len(points), 4))
    filtered = np.empty_like(predicted)
    predicted_cov = np.empty((len(points), 4, 4))
    filtered_cov = np.empty_like(predicted_cov)

    for i, point in enumerate(points):
        if i:
            state = transition @ state
            covariance = transition @ covariance @ transition.T + process_noise
        predicted[i], predicted_cov[i] = state, covariance
        gain = np.linalg.solve(observation @ covariance @ observation.T + measurement_noise,
                               (covariance @ observation.T).T).T
        state = state + gain @ (point - observation @ state)
        residual_map = np.eye(4) - gain @ observation
        covariance = residual_map @ covariance @ residual_map.T + gain @ measurement_noise @ gain.T
        filtered[i], filtered_cov[i] = state, covariance

    smoothed = filtered.copy()
    for i in range(len(points) - 2, -1, -1):
        gain = np.linalg.solve(predicted_cov[i + 1].T, (filtered_cov[i] @ transition.T).T).T
        smoothed[i] += gain @ (smoothed[i + 1] - predicted[i + 1])
    return smoothed

In [ ]:
starts = np.flatnonzero(ball_seen & ~np.r_[False, ball_seen[:-1]])
ends = np.flatnonzero(ball_seen & ~np.r_[ball_seen[1:], False])
signals = pd.DataFrame(np.nan, index=pd.Index(frame_ids, name="frame"),
                       columns=["acceleration_m_s2", "nearest_distance_m", "nearest_offset"])
signals["eligible"] = False
edge_guard = round(EDGE_GUARD_SECONDS * FPS)
for first, last in zip(starts, ends):
    idx = np.arange(first, last + 1)
    if len(idx) / FPS < MIN_RUN_SECONDS:
        continue
    state = smooth_ball_rts(ball_xy[idx], FPS, MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2)
    acceleration = np.linalg.norm(np.gradient(state[:, 2:], 1 / FPS, axis=0), axis=1)
    distances = np.linalg.norm(player_xy[idx] - state[:, None, :2], axis=2)
    distances = np.where(np.isfinite(distances), distances, np.inf)
    nearest_distance = distances.min(axis=1)
    nearest_offset = np.where(np.isfinite(nearest_distance), distances.argmin(axis=1), np.nan)
    nearest_distance[~np.isfinite(nearest_distance)] = np.nan
    signals.loc[idx, ["acceleration_m_s2", "nearest_distance_m", "nearest_offset"]] = np.c_[acceleration, nearest_distance, nearest_offset]
    if len(idx) > 2 * edge_guard:
        signals.loc[idx[edge_guard:-edge_guard], "eligible"] = True

assert signals.loc[~ball_seen, "acceleration_m_s2"].isna().all(), "Beschleunigungen dürfen Balllücken nicht überbrücken."
local_peak = signals["acceleration_m_s2"].gt(signals["acceleration_m_s2"].shift()) & signals["acceleration_m_s2"].ge(signals["acceleration_m_s2"].shift(-1))
signals["candidate"] = (signals["eligible"] & local_peak & signals["acceleration_m_s2"].ge(PEAK_ACCELERATION_M_S2)
                        & signals["nearest_distance_m"].le(NEAR_DISTANCE_M))
candidate_side = np.full(N_FRAMES, "", dtype="<U1")
candidate_mask = signals["candidate"].to_numpy()
candidate_side[candidate_mask] = tracks["side"].to_numpy()[signals.loc[candidate_mask, "nearest_offset"].to_numpy(dtype=int)]
print(f"{signals['eligible'].sum():,} Frames für die Peakdiagnose auswertbar; {candidate_mask.sum():,} Kandidaten im Gesamtvideo.")

## 4. Vom Kontaktkandidaten zu H/A/D

Die neue Regel wird vor dem Einlesen der GT vollständig festgelegt:

1. **D bleibt D:** Alle von der Baseline als D markierten Frames werden übernommen und setzen den Kontaktzustand zurück.
2. **Initialisierung:** Vor dem ersten verwendbaren Kandidaten eines Live-Abschnitts bleiben die H/A-Vorhersagen der Baseline erhalten.
3. **Kandidatenereignis:** Das Team des nächsten Spielers am Kandidatenframe übernimmt den Zustand H oder A.
4. **Beibehalten:** Ohne neues Ereignis bleibt dieses Team in Besitz, auch bei einer Balllücke. Ein gegnerischer Kandidat löst sofort einen Wechsel aus.
5. **Halbzeitgrenze:** Der Zustand wird zu Beginn jeder veröffentlichten Halbzeit neu initialisiert.

Das ist bewusst eine erste **Kontaktzustands-Heuristik**. Es gibt noch keine zusätzliche Mindestkontrolldauer, Passbestätigung oder Unterscheidung von Abprallern. Die D-Erkennung kann damit konstruktionsbedingt nicht besser werden. Auch innerhalb falsch als Live erkannten Unterbrechungen können H/A-Fehler bestehen bleiben.

In [ ]:
def classify_from_contacts(baseline, event_side, halves):
    prediction = baseline.copy()
    uses_contact_state = np.zeros(len(baseline), dtype=bool)
    for _, first, last in halves:
        last_team = None
        for frame in range(first, last + 1):
            if baseline[frame] == "D":
                last_team = None
                continue
            if event_side[frame] in ("H", "A"):
                last_team = event_side[frame]
            if last_team is not None:
                prediction[frame] = last_team
                uses_contact_state[frame] = True
    return prediction, uses_contact_state

contact_labels, uses_contact_state = classify_from_contacts(baseline_labels, candidate_side, HALVES)
assert np.array_equal(contact_labels == "D", baseline_labels == "D"), "Dieser Versuch darf die D-Erkennung nicht verändern."
coverage_rows = []
for half, first, last in HALVES:
    idx = np.arange(first, last + 1)
    live = baseline_labels[idx] != "D"
    coverage_rows.append({"Halbzeit": half, "Kandidaten": candidate_mask[idx].sum(),
                          "Kandidaten in Baseline-D (ignoriert)": (candidate_mask[idx] & ~live).sum(),
                          "Live-Frames aus Kontaktzustand (%)": 100 * uses_contact_state[idx][live].mean(),
                          "Geänderte Labels": (contact_labels[idx] != baseline_labels[idx]).sum()})
display(pd.DataFrame(coverage_rows).round(2))

## 5. Vorhersagen festschreiben und danach die öffentliche GT laden

Beide Methoden erzeugen eine vollständige CSV mit 199.319 Frames. Die neu geschriebene Vergleichsbaseline wird byteweise gegen die Originalreferenz geprüft. Erst danach lesen wir die öffentliche GT für den Scorevergleich ein. Außerhalb des öffentlichen Ausschnitts kennen wir keine Bewertungslabels und behaupten dort keine Leistung.

In [ ]:
predictions = {BASELINE_NAME: baseline_labels, METHOD_NAME: contact_labels}
for name, labels in predictions.items():
    pd.DataFrame({"frame": frame_ids, "label": labels}).to_csv(SUBMISSION_DIR / f"{name}.csv", index=False, lineterminator="\n")
with ZipFile(ARCHIVE) as official:
    assert (SUBMISSION_DIR / f"{BASELINE_NAME}.csv").read_bytes() == official.read(ARCHIVE_ROOT + f"evaluation/results/{BASELINE_NAME}.csv"), "Vergleichsbaseline wurde verändert."
    gt = pd.read_csv(BytesIO(official.read(ARCHIVE_ROOT + "evaluation/gt/BAR-ATH_possession_gt_public.csv")))
    public_meta = json.loads(official.read(ARCHIVE_ROOT + "evaluation/gt/BAR-ATH_possession_gt_public_meta.json"))
assert np.array_equal(gt["frame"], frame_ids), "Vorhersagen und GT sind nicht framegleich."
graded = gt["label"].ne("X").to_numpy()
public_frames = frame_ids[graded]
print(f"Öffentliche Evaluation: {graded.sum():,} Frames, {public_meta['n_possessions']} Ballbesitzphasen.")

## 6. Unveränderten offiziellen Validator und Evaluator verwenden

Beide CSVs werden mit derselben öffentlichen GT und denselben Standardeinstellungen wie in Notebook 02 bewertet. Wir ändern keine Metrik, werten keine privaten Labels aus und suchen nach diesem Vergleich keine neuen Parameter. `bootstrap=0` entspricht der bisherigen Referenzbewertung; kleine Scoreunterschiede werden nicht als statistisch abgesicherte Verbesserung interpretiert.

In [ ]:
validation_rows = []
with TemporaryDirectory(prefix="mtah04-official-") as temporary:
    with ZipFile(ARCHIVE) as official:
        official.extractall(temporary)
    official_root = Path(temporary) / ARCHIVE_ROOT
    for name in predictions:
        validation = subprocess.run([sys.executable, str(official_root / "evaluation/validate_submission.py"),
                                     str(SUBMISSION_DIR / f"{name}.csv")], capture_output=True, text=True, check=True)
        validation_rows.append({"Methode": name, "Validator": validation.stdout.splitlines()[0]})
    evaluation = subprocess.run(
        [sys.executable, str(official_root / "evaluation/run_evaluation.py"), "--gt", "public",
         "--results", str(SUBMISSION_DIR), "--out", str(EVALUATION_DIR), "--no-detail"],
        capture_output=True, text=True, check=True)
(EVALUATION_DIR / "official_stdout.txt").write_text(evaluation.stdout, encoding="utf-8")
display(pd.DataFrame(validation_rows))
print(evaluation.stdout)

## 7. Hauptmetriken und Richtung des Ergebnisses

Die Rangmetrik `spellF1` bewertet ganze H-/A-Phasen anhand zeitlicher Überlappung. `macroF1` ist die frameweise Ergänzung. Phasenanzahl und mittlere Dauer helfen, starke Fragmentierung oder zu langes Beibehalten zu erkennen; eine passende Anzahl allein ist kein Qualitätsnachweis. Alle Zahlen hier beziehen sich nur auf die öffentlichen fünf Minuten.

In [ ]:
scoreboard = pd.read_csv(EVALUATION_DIR / "scoreboard.csv")
results = scoreboard.set_index("submission").loc[list(predictions)]
display(results[["macro_f1", "spell_f1_mean", "poss", "dur", "f1_H", "f1_A", "f1_D"]].round(4))
delta = results.loc[METHOD_NAME, ["macro_f1", "spell_f1_mean"]] - results.loc[BASELINE_NAME, ["macro_f1", "spell_f1_mean"]]
print(f"Änderung gegenüber smooth50: macroF1 {delta['macro_f1']:+.4f}; spellF1 {delta['spell_f1_mean']:+.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
results[["macro_f1", "spell_f1_mean"]].rename(columns={"macro_f1": "macroF1", "spell_f1_mean": "spellF1"}).plot.bar(
    ax=axes[0], color=["#8aaac9", "#1764ab"], rot=12)
axes[0].set(title="Erster Klassifikationsvergleich: öffentlicher Ausschnitt", xlabel="", ylabel="Score", ylim=(0, 1))
axes[1].bar(results.index, results["poss"], color=["#8aaac9", "#1764ab"])
axes[1].axhline(public_meta["n_possessions"], color="#c83e41", linestyle="--", label="Ground Truth: 17 Phasen")
axes[1].tick_params(axis="x", rotation=12)
axes[1].set(title="Ballbesitzphasen im öffentlichen Ausschnitt", ylabel="Anzahl Phasen")
axes[1].legend()
plt.show()

## 8. Was verändert die neue Zuordnung konkret?

Der Zeitvergleich zeigt GT, unveränderte Vergleichsbaseline und Kontaktzustands-Heuristik. Die Konfusionsmatrizen darunter unterscheiden Teamverwechslungen von unverändert gebliebenen D-Fehlern. Wir dürfen nur eine Aussage über die getestete Heuristik treffen, nicht über alle denkbaren Kontrollmodelle aus dem Paper.

In [ ]:
rows = [gt.loc[graded, "label"].to_numpy()] + [labels[graded] for labels in predictions.values()]
label_to_number = {label: number for number, label in enumerate(LABEL_ORDER)}
timeline = np.array([[label_to_number[label] for label in row] for row in rows])
fig, ax = plt.subplots(figsize=(14, 3.2), constrained_layout=True)
ax.imshow(timeline, aspect="auto", interpolation="nearest", cmap=ListedColormap(COLORS), vmin=0, vmax=2,
          extent=(0, graded.sum() / FPS, 2.5, -0.5))
ax.set_yticks(range(3), ["Ground Truth", BASELINE_NAME, METHOD_NAME])
ax.set(title="Zeitverlauf: gleiche D-Erkennung, unterschiedliche H/A-Zuordnung", xlabel="Sekunden seit Beginn des öffentlichen Fensters")
ax.legend(handles=[Patch(color=color, label=label) for color, label in zip(COLORS, LABEL_ORDER)],
          loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=3)
plt.show()

In [ ]:
y_true = gt.loc[graded, "label"].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, (name, labels) in zip(axes, predictions.items()):
    confusion = pd.crosstab(pd.Categorical(y_true, categories=LABEL_ORDER),
                            pd.Categorical(labels[graded], categories=LABEL_ORDER), dropna=False)
    percent = confusion.div(confusion.sum(axis=1), axis=0) * 100
    image = ax.imshow(percent, cmap="Blues", vmin=0, vmax=100)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{confusion.iloc[i, j]}\n{percent.iloc[i, j]:.1f} %", ha="center", va="center",
                    color="white" if percent.iloc[i, j] > 55 else "black")
    ax.set(title=name, xticks=range(3), xticklabels=LABEL_ORDER, yticks=range(3), yticklabels=LABEL_ORDER,
           xlabel="Vorhersage", ylabel="Ground Truth")
fig.colorbar(image, ax=axes, label="Anteil innerhalb der GT-Klasse (%)")
plt.show()

## 9. Verbesserte und verschlechterte Frames getrennt zeigen

Die folgende Zerlegung ist eine deskriptive Fehleranalyse auf dem öffentlichen Ausschnitt. Sie ersetzt keine Phasenbewertung. Sie zeigt, ob ein veränderter Frame von falsch zu richtig, richtig zu falsch oder zwischen zwei falschen Labels wechselt. Dadurch wird eine mögliche Verbesserung nicht nur anhand einzelner günstiger Szenen beurteilt.

In [ ]:
base_public, contact_public = baseline_labels[graded], contact_labels[graded]
base_correct, contact_correct = base_public == y_true, contact_public == y_true
changed = base_public != contact_public
error_changes = pd.Series({
    "Unveränderte Vorhersagen": int((~changed).sum()),
    "Falsch → richtig": int((~base_correct & contact_correct).sum()),
    "Richtig → falsch": int((base_correct & ~contact_correct).sum()),
    "Falsch → anderes falsches Label": int((changed & ~base_correct & ~contact_correct).sum()),
}, name="Öffentliche Frames")
display(error_changes.to_frame())
public_diagnostics = pd.DataFrame({"frame": public_frames, "GT": y_true, "baseline": base_public,
                                   "contact_state": contact_public, "contact_state_used": uses_contact_state[graded]})
pair_changes = public_diagnostics[["GT", "baseline", "contact_state"]].ne(
    public_diagnostics[["GT", "baseline", "contact_state"]].shift()).any(axis=1).cumsum()
sections = public_diagnostics.groupby(pair_changes).agg(
    start_frame=("frame", "first"), end_frame=("frame", "last"), GT=("GT", "first"),
    baseline=("baseline", "first"), contact_state=("contact_state", "first"), n_frames=("frame", "size"))
sections = sections.loc[sections["baseline"].ne(sections["contact_state"])].copy()
sections["duration_s"] = sections["n_frames"] / FPS
sections["effect"] = np.where(sections["GT"].eq(sections["contact_state"]), "verbessert",
                               np.where(sections["GT"].eq(sections["baseline"]), "verschlechtert", "weiterhin falsch"))
display(sections.nlargest(8, "duration_s").drop(columns="n_frames").round(2))

## 10. Aktuelles Ergebnis festhalten

Es wurde genau eine neue Klassifikationsregel geprüft. Die Unterbrechungserkennung, Diagnoseparameter, öffentliche Bewertung und Vergleichsbaseline wurden unverändert gehalten. Ein Ergebnis auf 17 Ballbesitzphasen ist explorativ und keine belastbare Aussage über die Qualität auf dem vollständigen Spiel.

In [ ]:
spell_change = float(delta["spell_f1_mean"])
direction = "höher" if spell_change > 0 else "niedriger" if spell_change < 0 else "gleich"
display(Markdown(
    f"**Beobachtung:** Der öffentliche spellF1 der Kontaktzustands-Heuristik beträgt **{results.loc[METHOD_NAME, 'spell_f1_mean']:.4f}**, "
    f"gegenüber **{results.loc[BASELINE_NAME, 'spell_f1_mean']:.4f}** für smooth50. Er ist damit in diesem Versuch **{direction}**. "
    f"Der macroF1 beträgt **{results.loc[METHOD_NAME, 'macro_f1']:.4f}** gegenüber **{results.loc[BASELINE_NAME, 'macro_f1']:.4f}**. "
    "Die D-Maske ist exakt identisch; ein trainierter Kontrollklassifikator wurde nicht getestet."))
manifest = {"official_commit": COMMIT, "archive_sha256": ARCHIVE_SHA256, "python": sys.version.split()[0],
            "packages": {name: version(name) for name in ["numpy", "pandas", "matplotlib", "IPython"]},
            "comparison": list(predictions), "n_frames": N_FRAMES, "fps": FPS, "pitch_m": PITCH.tolist(),
            "measurement_sigma_m": MEASUREMENT_SIGMA_M, "process_sigma_m_s2": PROCESS_SIGMA_M_S2,
            "near_distance_m": NEAR_DISTANCE_M, "peak_acceleration_m_s2": PEAK_ACCELERATION_M_S2,
            "min_run_seconds": MIN_RUN_SECONDS, "edge_guard_seconds": EDGE_GUARD_SECONDS,
            "dead_ball_rule": "unchanged baseline_smooth50", "initialization": "baseline until first candidate in each live block",
            "ground_truth": "public", "bootstrap": 0, "parameter_search": False}
(OUTPUT_DIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
error_changes.to_csv(OUTPUT_DIR / "public_error_changes.csv")
print(f"Vollständige Vorhersagen: {SUBMISSION_DIR}")
print(f"Offizielle Bewertung: {EVALUATION_DIR}")

**Weitere Beobachtung im ausgeführten Vergleich:** Die neue Regel erzeugt 28 H-/A-Phasen, die Baseline 26, die öffentliche GT 17. Der höhere spellF1 entsteht also nicht durch eine näher an der GT liegende Gesamtzahl. Der H-F1 steigt von 0,7781 auf 0,8116, während der A-F1 von 0,7615 auf 0,7430 fällt. Die Gesamtverbesserung verteilt sich nicht gleichmäßig auf die Klassen.

**Interpretation und nächste Entscheidung:** Ein positives oder negatives Ergebnis gilt zunächst nur für diese konkrete Regel. Kontaktindizien und kontrollierter Besitz sind verschiedene Sachverhalte. Falsche Kandidaten können Besitzwechsel erfinden; übersehene Kandidaten können einen alten Zustand zu lange fortsetzen. Fehler der unveränderten D-Erkennung können durch diesen Versuch nicht behoben werden.

Als nächster Schritt wäre eine gezielte Videosichtung der größten neu entstandenen und behobenen Fehler sinnvoll. Sie könnte zeigen, ob eher die Kandidatenerkennung, die Zuordnung zum nächsten Spieler oder das sofortige Umschalten bei jedem gegnerischen Kontakt problematisch ist. Eine zusätzliche Kontrollbestätigung oder geänderte Unterbrechungserkennung wären weitere methodische Änderungen und wurden hier nicht automatisch ergänzt.

Quellen: [Paper](https://doi.org/10.1371/journal.pone.0179953), [offizielle Baseline](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/starter/baseline.py), [offizielle Evaluation](https://github.com/metrica-sports/more-than-a-hack-2027/blob/3de6b60aeabd2475d95879a651d38b72bc4fba00/EVALUATION.md). Dies ist eine eigenständige Heuristik, keine Reproduktion des trainierten Kontrollmodells aus dem Paper.